In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

from __future__ import annotations

from pathlib import Path

import pandas as pd
from PIL import Image
from fiftyone.core.session.notebooks import display
from torch.utils.data import Dataset

In [3]:
class BeerDataset(Dataset):

    PATH_PRIORITY = ("path", "aug_path", "cropped_path", "orig_path")

    def __init__(
        self,
        meta_csv: str | Path = "data/meta/full_dataset.csv",
        root: str | Path | None = None,
        transform=None,
        target_transform=None,
        return_info: bool = False,
        path_columns: tuple[str, ...] | None = None,
    ):
        self.meta_csv = Path(meta_csv)
        if not self.meta_csv.exists():
            raise FileNotFoundError(f"Metadata CSV not found: {self.meta_csv}")

        self.root = Path(root) if root is not None else Path(".")
        self.transform = transform
        self.target_transform = target_transform
        self.return_info = return_info

        df = pd.read_csv(self.meta_csv)
        required_cols = {"klass"}
        missing = required_cols - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns in metadata: {sorted(missing)}")

        candidates = path_columns or self.PATH_PRIORITY
        available = tuple(col for col in candidates if col in df.columns)
        if not available:
            raise ValueError(
                "None of the provided path columns were found in the metadata."
            )

        resolved_path = df[available[0]].copy()
        for col in available[1:]:
            resolved_path = resolved_path.fillna(df[col])

        df = df.assign(resolved_path=resolved_path)
        df = df.dropna(subset=["resolved_path"]).reset_index(drop=True)
        if df.empty:
            raise ValueError("No usable entries found in the metadata file.")

        df["resolved_path"] = df["resolved_path"].astype(str)
        self._path_column = "resolved_path"
        self._records = df

        classes = sorted(df["klass"].unique())
        self.class_to_idx = {klass: idx for idx, klass in enumerate(classes)}
        self.idx_to_class = {idx: klass for klass, idx in self.class_to_idx.items()}

    def __len__(self) -> int:
        return len(self._records)

    def _resolve_path(self, rel_path: str) -> Path:
        p = Path(rel_path)
        if p.is_absolute():
            return p
        return (self.root / p).resolve()

    def __getitem__(self, index: int):
        row = self._records.iloc[index]
        img_path = self._resolve_path(row[self._path_column])
        if not img_path.exists():
            raise FileNotFoundError(f"Image not found on disk: {img_path}")

        image = Image.open(img_path).convert("RGB")
        label = self.class_to_idx[row["klass"]]

        if self.transform is not None:
            image = self.transform(image)
        if self.target_transform is not None:
            label = self.target_transform(label)

        if self.return_info:
            info = row.to_dict() | {"img_path": str(img_path)}
            return image, label, info
        return image, label

In [4]:
class AlexNetClassifier(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=0),   # Conv1
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),                   # Overlapping pooling
            
            nn.Conv2d(96, 256, kernel_size=5, padding=2),            # Conv2
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            nn.Conv2d(256, 384, kernel_size=3, padding=1),           # Conv3
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1),           # Conv4
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),           # Conv5
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),                                       # Dropout (регуляризація)
            nn.Linear(256 * 6 * 6, 4096),                            # FC6
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),                                   # FC7
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes)                             # FC8 — вихід для Beer класів
        )

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, mean=0.0, std=0.01)
                nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.01)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


In [5]:
class DetectionHead(nn.Module):
    def __init__(self, in_channels, num_classes, num_anchors, dropout_p=0.5):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 512, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout_p)
        self.cls_head = nn.Conv2d(512, num_anchors * num_classes, kernel_size=1)
        self.box_head = nn.Conv2d(512, num_anchors * 4, kernel_size=1)

        # Ініціалізація
        for m in [self.conv, self.cls_head, self.box_head]:
            nn.init.normal_(m.weight, 0, 0.01)
            nn.init.constant_(m.bias, 0)

    def forward(self, feat):
        x = self.relu(self.conv(feat))
        x = self.dropout(x)
        cls = self.cls_head(x)
        box = self.box_head(x)
        return cls, box

In [6]:
class AlexNetDetector(nn.Module):
    def __init__(self, num_classes=21, num_anchors=9):
        super().__init__()
        self.backbone = AlexNetBackbone()
        self.head = DetectionHead(in_channels=256, num_classes=num_classes, num_anchors=num_anchors)

    def forward(self, x):
        feat = self.backbone(x)
        cls, box = self.head(feat)
        return cls, box

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_ds = BeerDataset(
    meta_csv="data/meta/full_dataset.csv",
    root="data/",
    transform=train_tfms
)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)

num_classes = len(train_ds.class_to_idx)
model = AlexNetClassifier(num_classes=num_classes)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,          # learning rate
    momentum=0.9,     # momentum
    weight_decay=5e-4 # L2 regularization
)
criterion = nn.CrossEntropyLoss()


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(3):  
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
    
    print(f"Epoch [{epoch+1}/3] | Loss: {running_loss/len(train_loader.dataset):.4f}")

Мережа тренувалася з використанням стохастичного градієнтного спуску (stochastic gradient descent).

- Розмір пакету (Batch Size): 128 прикладів.
    - Значення: Це кількість навчальних прикладів, які використовуються для обчислення середнього градієнта та оновлення ваг за одну ітерацію.

- Імпульс (Momentum, v): 0.9.
    - Значення: Використовується у правилі оновлення ваг для прискорення градієнтного спуску. Правило оновлення змінної імпульсу v 
i+1
​
  включає множення попереднього значення на 0.9.
- Зменшення ваги (Weight Decay): 0.0005.
    - Значення: Це параметр регуляризації (по суті, L2-регуляризація), який додається до функції втрат. Він був важливим не лише як регуляризатор, але й для того, щоб модель взагалі могла навчитися, оскільки він зменшує помилку навчання моделі.
- Швидкість навчання (Learning Rate, ϵ): Ініціалізована на 0.01.
    - Значення: Визначає розмір кроку, зробленого під час оптимізації. Швидкість навчання регулювалася вручну протягом навчання. Евристика полягала в тому, щоб поділити швидкість навчання на 10, коли рівень помилки валідації переставав покращуватися при поточній швидкості.
Гіперпараметри регуляризації
Для боротьби з перенавчанням (overfitting), оскільки мережа мала 60 мільйонів параметрів, використовувалися такі методи та їхні параметри:
- Dropout (Викидання): Імовірність 0.5.
    - Значення: Під час навчання вихід кожного прихованого нейрона обнуляється з імовірністю 0.5. Це змушує нейрони вивчати більш стійкі ознаки, які є корисними у поєднанні з багатьма різними випадковими підмножинами інших нейронів, тим самим зменшуючи складну коадаптацію нейронів. Dropout застосовувався у перших двох повністю зв'язаних шарах.
- Аугментація даних (Data Augmentation) – Варіації інтенсивності RGB: Параметр, пов'язаний зі зміною інтенсивності каналів RGB, використовує випадкову змінну, взяту з розподілу Гауса з нульовим середнім та стандартним відхиленням 0.1.
    - Значення: Ця схема приблизно фіксує властивість природних зображень про те, що ідентичність об'єкта є інваріантною до змін інтенсивності та кольору освітлення. Це знизило помилку top-1 більш ніж на 1%.
Гіперпараметри ініціалізації
Ваги та зміщення нейронів були ініціалізовані за допомогою наступних констант:
- Ініціалізація ваг: Ваги у кожному шарі ініціалізувалися з розподілу Гауса з нульовим середнім та стандартним відхиленням 0.01.
- Ініціалізація зміщень (Bias):
    - У другому, четвертому та п'ятому згорткових шарах, а також у прихованих повністю зв'язаних шарах використовувалася константа 1.
    - У решті шарів використовувалася константа 0.
    - Значення: Ініціалізація зміщень одиницею прискорює ранні етапи навчання, забезпечуючи позитивні вхідні дані для нейронів ReLU (Rectified Linear Units).
Гіперпараметри архітектурних компонентів
Локальна нормалізація відгуку (Local Response Normalization, LRN)
Ця схема використовує чотири константи, значення яких були визначені за допомогою валідаційного набору:
- k (константа зміщення): k=2.
- n (глибина нормалізації): n=5 (сума відбувається по n «суміжних» картах ядер).
- α (коефіцієнт масштабування): α=10 
−4
 .
- β (експонента): β=0.75.
    - Значення: Ці константи регулюють рівняння нормалізації відгуку. Ця нормалізація імітує форму бічного гальмування, створюючи конкуренцію між виходами нейронів, обчисленими з використанням різних ядер, і сприяє узагальненню.
Згорткові шари (Конкретні параметри шарів)
Хоча це більше деталі архітектури, ніж гіперпараметри навчання, вони контролюють розмір вихідних даних і підлягають налаштуванню:
- Stride (Крок) та Розмір ядра (Kernel Size) Першого шару: Перший згортковий шар фільтрує вхідне зображення 224×224×3 за допомогою 96 ядер розміром 11×11×3 із кроком 4 пікселі.
    - Значення: Крок (stride) визначає відстань між центрами рецептивних полів сусідніх нейронів.
Перекривне об'єднання (Overlapping Pooling)
Ця мережа використовує перекривне об'єднання:
- Розмір області об'єднання (z): z=3 (розмір околу 3×3).
- Крок об'єднання (s): s=2 (відстань між центрами сусідніх блоків).
    - Значення: Оскільки s<z (2 < 3), відбувається перекриття. Використання перекривного об'єднання (порівняно з традиційним неперекривним s=2,z=2) зменшило помилки top-1 та top-5.